In [1]:
import os
import sys
import numpy as np
import pandas as pd

In [2]:
root_path = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if root_path not in sys.path:
    sys.path.append(root_path)

from src.optimizers.adam import AdamOptimizer
from src.models.quantile_regression import QuantileRegressionV1

In [3]:
import numpy as np
import pandas as pd

train_path = r"D:\STUDY\25-26\HK2\toan_toi_uu\toanToiUu\dataset\foodDeli_processed\train_processed.csv"

# Đọc file và loại bỏ luôn các dòng bị khuyết dữ liệu
train = pd.read_csv(train_path)

target_column = 'Time_taken' 
columns_to_drop = ['ID', target_column]

np.random.seed(42)
shuffled_indices = np.random.permutation(len(train))

# Chia tỷ lệ 80% Train - 20% Test
train_size = int(len(train) * 0.8)

# Tách chính xác các mảng chỉ mục
train_indices = shuffled_indices[:train_size]
test_indices   = shuffled_indices[train_size:]

# SỬA LỖI Ở ĐÂY: Trích xuất từ df_goc ban đầu để không bị ghi đè
df_trainn = train.iloc[train_indices]
df_test  = train.iloc[test_indices]

# Trích xuất ma trận đặc trưng
X_train_raw = df_trainn.drop(columns=columns_to_drop, errors='ignore').values
y_train     = df_trainn[target_column].values

X_test_raw  = df_test.drop(columns=columns_to_drop, errors='ignore').values
y_test      = df_test[target_column].values

# Thêm cột hệ số chặn Bias
def add_bias_column(X):
    return np.hstack((np.ones((X.shape[0], 1)), X))

X_train = add_bias_column(X_train_raw)
X_test  = add_bias_column(X_test_raw)

print(f"Kích thước Train X: {X_train.shape} | Vector y: {y_train.shape}")
print(f"Kích thước Test X:  {X_test.shape}  | Vector y: {y_test.shape}")

Kích thước Train X: (30043, 23) | Vector y: (30043,)
Kích thước Test X:  (7511, 23)  | Vector y: (7511,)


In [4]:
adam_optimizer = AdamOptimizer(learning_rate=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8)
model = QuantileRegressionV1(tau=0.9, optimizer=adam_optimizer)

print("Bắt đầu huấn luyện Batch Gradient Descent với Adam")
loss_history = model.fit(X_train, y_train, epochs=50000,  )
print("Thuật toán tối ưu hóa hoàn tất!")

Bắt đầu huấn luyện Batch Gradient Descent với Adam
Epoch 0/50000, Loss: 23.8476
Epoch 100/50000, Loss: 19.0484
Epoch 200/50000, Loss: 14.3521
Epoch 300/50000, Loss: 10.0976
Epoch 400/50000, Loss: 7.0092
Epoch 500/50000, Loss: 5.0699
Epoch 600/50000, Loss: 3.9290
Epoch 700/50000, Loss: 3.2438
Epoch 800/50000, Loss: 2.8120
Epoch 900/50000, Loss: 2.5287
Epoch 1000/50000, Loss: 2.3378
Epoch 1100/50000, Loss: 2.2048
Epoch 1200/50000, Loss: 2.1111
Epoch 1300/50000, Loss: 2.0431
Epoch 1400/50000, Loss: 1.9922
Epoch 1500/50000, Loss: 1.9518
Epoch 1600/50000, Loss: 1.9184
Epoch 1700/50000, Loss: 1.8901
Epoch 1800/50000, Loss: 1.8648
Epoch 1900/50000, Loss: 1.8422
Epoch 2000/50000, Loss: 1.8213
Epoch 2100/50000, Loss: 1.8014
Epoch 2200/50000, Loss: 1.7822
Epoch 2300/50000, Loss: 1.7639
Epoch 2400/50000, Loss: 1.7460
Epoch 2500/50000, Loss: 1.7291
Epoch 2600/50000, Loss: 1.7128
Epoch 2700/50000, Loss: 1.6974
Epoch 2800/50000, Loss: 1.6827
Epoch 2900/50000, Loss: 1.6685
Epoch 3000/50000, Loss: 1.6

In [5]:
y_pred_test = model.predict(X_test)
test_loss = model.pinball_loss(y_test, y_pred_test)
print(f"Pinball Loss trên tập Test:       {test_loss:.4f}")

Pinball Loss trên tập Test:       1.0786


In [6]:
weights_dir = r"D:\STUDY\25-26\HK2\toan_toi_uu\toanToiUu\results\weights"
weights_path = os.path.join(weights_dir, "adam_theta.npy")
loss_path  = os.path.join(weights_dir, "adam_loss.npy")
# Sử dụng numpy để lưu trữ cấu trúc mảng
np.save(weights_path, model.theta)
np.save(loss_path, np.array(loss_history))

print(f"Trọng số tối ưu đã được lưu thành công! (theta):\n", model.theta)

Trọng số tối ưu đã được lưu thành công! (theta):
 [ 4.44976691e+01  2.40521990e+00 -2.18256737e+00  1.61234562e+00
 -1.16011585e-02  1.33512877e-02 -4.76275880e+00 -4.90413236e+00
 -5.19537006e+00 -4.74768206e+00  1.74665000e+00 -7.30878934e+00
 -1.84169289e+00 -2.24749263e-01  9.06475342e-02  1.84863807e-01
 -1.23422616e-01  2.99416374e-02 -6.10438249e+00 -6.17898514e+00
  7.81596767e+00  8.64803248e+00 -1.58386413e+00]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.style.use('seaborn-v0_8-whitegrid') 
plt.plot(loss_history, label='Adam Loss', color='#e74c3c', linewidth=2.5)
plt.title('Loss Adam Optimizer', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Epochs', fontsize=12, fontweight='bold')
plt.ylabel('Pinball Loss', fontsize=12, fontweight='bold')
plt.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.show()